Name: Rushikesh Anil Khairnar  
Student: 5115301  
Roll NO: 21  
College: B. K. Birla College

# **Practical 7**
**1.Design an ETL pipeline to extract data from a CSV file, clean it, transform it, and store it in a database.**



In [1]:
import csv
import sqlite3
import os

# 1. SETUP: Create sample raw CSV
csv_file = 'raw_data.csv'
db_file = 'etl_task1.db'
with open(csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'name', 'salary'])
    writer.writerow(['1', 'alice smith', '50000'])
    writer.writerow(['2', '', '40000']) # Missing name (needs cleaning)
    writer.writerow(['3', 'bob jones', '60000'])

print("--- Task 1: CSV ETL Pipeline ---")

# 2. EXTRACT
raw_data = []
with open(csv_file, 'r') as f:
    reader = csv.DictReader(f)
    raw_data = list(reader)

# 3. CLEAN & TRANSFORM
clean_data = []
for row in raw_data:
    if not row['name'].strip():
        continue # Clean: Skip missing names

    # Transform: Capitalize name and calculate a 10% tax column
    transformed_row = (
        int(row['id']),
        row['name'].title(),
        float(row['salary']),
        float(row['salary']) * 0.10
    )
    clean_data.append(transformed_row)

# 4. LOAD
conn = sqlite3.connect(db_file)
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS employees
                  (id INTEGER, name TEXT, salary REAL, tax REAL)''')
cursor.executemany('INSERT INTO employees VALUES (?, ?, ?, ?)', clean_data)
conn.commit()

# 5. VISIBLE OUTPUT
cursor.execute('SELECT * FROM employees')
print("Database contents after ETL:")
for row in cursor.fetchall():
    print(row)

# Cleanup
conn.close()
os.remove(csv_file)
os.remove(db_file)

--- Task 1: CSV ETL Pipeline ---
Database contents after ETL:
(1, 'Alice Smith', 50000.0, 5000.0)
(3, 'Bob Jones', 60000.0, 6000.0)


**2.Extract data from multiple CSV files and combine them into a single dataset.**


In [2]:
import csv
import os

# 1. SETUP: Create multiple CSVs
files = ['region_a.csv', 'region_b.csv']
with open(files[0], 'w', newline='') as f:
    csv.writer(f).writerows([['id', 'product'], ['101', 'Laptop'], ['102', 'Mouse']])
with open(files[1], 'w', newline='') as f:
    csv.writer(f).writerows([['id', 'product'], ['103', 'Keyboard'], ['104', 'Monitor']])

print("\n--- Task 2: Combine Multiple CSVs ---")

# 2. EXTRACT AND COMBINE
combined_dataset = []
header_extracted = False

for file in files:
    with open(file, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)
        if not header_extracted:
            combined_dataset.append(header)
            header_extracted = True

        # Append data rows
        for row in reader:
            combined_dataset.append(row)

# 3. VISIBLE OUTPUT
print("Combined Dataset:")
for row in combined_dataset:
    print(row)

# Cleanup
for file in files:
    os.remove(file)


--- Task 2: Combine Multiple CSVs ---
Combined Dataset:
['id', 'product']
['101', 'Laptop']
['102', 'Mouse']
['103', 'Keyboard']
['104', 'Monitor']


**3.Build a pipeline to extract JSON data, transform selected fields, and load it into a relational database.**


In [3]:
import json
import sqlite3
import os

# 1. SETUP: Create JSON file
json_file = 'orders.json'
db_file = 'etl_task3.db'
data = [
    {"order_id": "A1", "details": {"customer": "John", "total_cents": 2500}, "date": "2026-10-06"},
    {"order_id": "A2", "details": {"customer": "Jane", "total_cents": 5000}, "date": "2026-10-07"}
]
with open(json_file, 'w') as f:
    json.dump(data, f)

print("\n--- Task 3: JSON ETL Pipeline ---")

# 2. EXTRACT
with open(json_file, 'r') as f:
    json_data = json.load(f)

# 3. TRANSFORM
transformed_data = []
for item in json_data:
    # Flatten nested JSON, convert cents to dollars, format date string
    transformed_data.append((
        item['order_id'],
        item['details']['customer'],
        item['details']['total_cents'] / 100.0, # Transform to dollars
        item['date'].replace("-", "/")        # Transform date format
    ))

# 4. LOAD
conn = sqlite3.connect(db_file)
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS orders
                  (order_id TEXT, customer TEXT, total_dollars REAL, order_date TEXT)''')
cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?)', transformed_data)
conn.commit()

# 5. VISIBLE OUTPUT
cursor.execute('SELECT * FROM orders')
print("Database contents after JSON ETL:")
for row in cursor.fetchall():
    print(row)

# Cleanup
conn.close()
os.remove(json_file)
os.remove(db_file)


--- Task 3: JSON ETL Pipeline ---
Database contents after JSON ETL:
('A1', 'John', 25.0, '2026/10/06')
('A2', 'Jane', 50.0, '2026/10/07')


**4.Design an ETL process to identify and remove invalid records.**


In [4]:
print("\n--- Task 4: Remove Invalid Records ---")

# 1. EXTRACT (Simulated raw data)
raw_users = [
    {"user_id": 1, "age": 25, "email": "alice@test.com"},
    {"user_id": 2, "age": -5, "email": "bob@test.com"}, # Invalid age
    {"user_id": 3, "age": 30, "email": "charlie_no_at_symbol.com"}, # Invalid email
    {"user_id": 4, "age": 40, "email": "david@test.com"}
]

valid_records = []
invalid_records = []

# 2. CLEAN (Identify and split)
for user in raw_users:
    is_valid = True
    reason = []

    if user['age'] < 0 or user['age'] > 120:
        is_valid = False
        reason.append("Age out of bounds")

    if "@" not in user['email']:
        is_valid = False
        reason.append("Invalid email format")

    if is_valid:
        valid_records.append(user)
    else:
        user['error_reason'] = " | ".join(reason)
        invalid_records.append(user)

# 3. VISIBLE OUTPUT
print("Valid Records Retained:")
for r in valid_records:
    print(f"  {r}")

print("\nInvalid Records Removed:")
for r in invalid_records:
    print(f"  {r}")


--- Task 4: Remove Invalid Records ---
Valid Records Retained:
  {'user_id': 1, 'age': 25, 'email': 'alice@test.com'}
  {'user_id': 4, 'age': 40, 'email': 'david@test.com'}

Invalid Records Removed:
  {'user_id': 2, 'age': -5, 'email': 'bob@test.com', 'error_reason': 'Age out of bounds'}
  {'user_id': 3, 'age': 30, 'email': 'charlie_no_at_symbol.com', 'error_reason': 'Invalid email format'}


**5.Create a pipeline that performs data validation before loading data into the target database.**


In [5]:
import sqlite3
import os

print("\n--- Task 5: Pre-Load Validation ---")

db_file = 'etl_task5.db'
conn = sqlite3.connect(db_file)
cursor = conn.cursor()
cursor.execute('CREATE TABLE products (id INTEGER, name TEXT, price REAL)')

incoming_data = [
    (1, "Table", 150.0),
    (2, "Chair", -20.0), # Fails price validation
    (3, None, 50.0)      # Fails name validation
]

# Validation Function
def validate_product(record):
    product_id, name, price = record
    if not name:
        return False, "Missing name"
    if price < 0:
        return False, "Negative price"
    return True, "Valid"

# Process and Load
successful_inserts = 0
for record in incoming_data:
    is_valid, msg = validate_product(record)
    if is_valid:
        cursor.execute('INSERT INTO products VALUES (?, ?, ?)', record)
        successful_inserts += 1
    else:
        print(f"Validation Failed for record {record}: {msg}")

conn.commit()

# VISIBLE OUTPUT
print(f"\nSuccessfully loaded {successful_inserts} valid records.")
cursor.execute('SELECT * FROM products')
for row in cursor.fetchall():
    print(row)

# Cleanup
conn.close()
os.remove(db_file)


--- Task 5: Pre-Load Validation ---
Validation Failed for record (2, 'Chair', -20.0): Negative price
Validation Failed for record (3, None, 50.0): Missing name

Successfully loaded 1 valid records.
(1, 'Table', 150.0)


**6.Implement incremental data loading instead of loading the entire dataset every time.**

In [6]:
import sqlite3
import os

print("\n--- Task 6: Incremental Loading ---")

db_file = 'etl_task6.db'
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# 1. SETUP: Create table and load initial historical data
cursor.execute('''CREATE TABLE inventory
                  (item_id INTEGER PRIMARY KEY, item_name TEXT, qty INTEGER)''')
cursor.execute("INSERT INTO inventory VALUES (1, 'Pen', 100)")
cursor.execute("INSERT INTO inventory VALUES (2, 'Pencil', 200)")
conn.commit()

print("Initial Database State:")
for row in cursor.execute('SELECT * FROM inventory').fetchall():
    print(f"  {row}")

# 2. EXTRACT: New batch of data (Contains 1 update, 1 new item, 1 unchanged)
new_batch = [
    (1, 'Pen', 100),    # Unchanged
    (2, 'Pencil', 250), # Incremental Update (Quantity changed)
    (3, 'Eraser', 50)   # Incremental Insert (New item)
]

# 3. INCREMENTAL LOAD (UPSERT Strategy)
# INSERT OR REPLACE automatically updates the row if the PRIMARY KEY (item_id) already exists.
cursor.executemany('''
    INSERT OR REPLACE INTO inventory (item_id, item_name, qty)
    VALUES (?, ?, ?)
''', new_batch)
conn.commit()

# 4. VISIBLE OUTPUT
print("\nDatabase State After Incremental Load (Notice ID 2 updated, ID 3 added, no duplicates):")
for row in cursor.execute('SELECT * FROM inventory').fetchall():
    print(f"  {row}")

# Cleanup
conn.close()
os.remove(db_file)


--- Task 6: Incremental Loading ---
Initial Database State:
  (1, 'Pen', 100)
  (2, 'Pencil', 200)

Database State After Incremental Load (Notice ID 2 updated, ID 3 added, no duplicates):
  (1, 'Pen', 100)
  (2, 'Pencil', 250)
  (3, 'Eraser', 50)
